In [1]:
import torch
import pandas as pd

# Pre-processing of the datasets

In [2]:
# Load datasets
train_df = pd.read_csv("/kaggle/input/datasets/faruqolasupo/moremi/train_qa.csv")
test_df = pd.read_csv("/kaggle/input/datasets/faruqolasupo/moremi/test_questions.csv")

In [5]:
# Read the first rows of the  test data
test_df.head()

,QuestionId,question,topic,care_setting,population
0,1001,How is pregnancy anaemia managed?,maternal_health,primary_care,pregnant
1,1002,Fever in a two-month-old — what to do?,emergency_triage,hospital,infant
2,1003,Pregnancy symptoms needing urgent review?,maternal_health,primary_care,pregnant
3,1004,Teen refuses school citing panic — approach?,mental_health_basics,school,child
4,1005,How can families prevent malaria?,infectious_disease,community,general


In [6]:
# Read the first rows of the column data
train_df.head()

,question,topic,care_setting,population,document_id,reference_answer,QuestionId
0,BP readings improved — skip tablets this weekend?,chronic_disease,primary_care,adult,doc_chr_002,No — never stop antihypertensives abruptly wit...,1
1,Pregnant and always tired — common workup?,maternal_health,primary_care,pregnant,doc_mat_003,Screen for anaemia and treat with iron plus di...,2
2,When should someone with flu-like symptoms sta...,infectious_disease,community,general,doc_inf_002,While febrile to reduce spread at school or work.,3
3,How do I manage type 2 diabetes day to day?,chronic_disease,primary_care,adult,doc_chr_001,"Balance meals, stay active, take medicines as ...",4
4,Feverish toddler — paracetamol cautions?,medication_safety,home,child,doc_med_003,Use weight-based doses and avoid exceeding dai...,5


In [7]:
# Check for missing data in the test data
test_df.isnull().sum()

QuestionId      0
question        0
topic           0
care_setting    0
population      0
dtype: int64

In [8]:
# Check for missing data in the training data
train_df.isnull().sum()

question            0
topic               0
care_setting        0
population          0
document_id         0
reference_answer    0
QuestionId          0
dtype: int64

# Tokenisation of datasets

We will be making use of special tokens
<BOS>
<QUESTION>
<ANSWER>
<EOS>
<PAD>
<UNK>

In [3]:
import re
import json

In [4]:
special_tokens = [
    "<PAD>",
    "<UNK>",
    "<BOS>",
    "<EOS>",
    "<QUESTION>",
    "<ANSWER>"
]


class SimpleTokenizer:

    def __init__(self):
        self.token_to_id = {}
        self.id_to_token = {}

    def tokenize(self, text):
        text = str(text).strip().lower()

        # Separate punctuation
        tokens = re.findall(
            r"\w+|[^\w\s]",
            text,
            flags=re.UNICODE
        )

        return tokens

    def build_vocab(self, texts):

        vocabulary = set()

        for text in texts:
            vocabulary.update(self.tokenize(text))

        all_tokens = special_tokens + sorted(vocabulary)

        self.token_to_id = {
            token: idx
            for idx, token in enumerate(all_tokens)
        }

        self.id_to_token = {
            idx: token
            for token, idx in self.token_to_id.items()
        }

    def encode(self, text):

        tokens = self.tokenize(text)

        unk_id = self.token_to_id["<UNK>"]

        return [
            self.token_to_id.get(token, unk_id)
            for token in tokens
        ]

    def decode(self, token_ids):

        tokens = [
            self.id_to_token.get(token_id, "<UNK>")
            for token_id in token_ids
        ]

        return " ".join(tokens)

    @property
    def vocab_size(self):
        return len(self.token_to_id)

    def save(self, path):

        data = {
            "token_to_id": self.token_to_id
        }

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    def load(self, path):

        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        self.token_to_id = data["token_to_id"]

        self.id_to_token = {
            int(idx): token
            for token, idx in self.token_to_id.items()
        }

# Prepare the training sequence

In [5]:
from torch.utils.data import Dataset


class QADataset(Dataset):

    def __init__(
        self,
        dataframe,
        tokenizer,
        max_length
    ):

        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

        self.pad_id = tokenizer.token_to_id["<PAD>"]
        self.bos_id = tokenizer.token_to_id["<BOS>"]
        self.eos_id = tokenizer.token_to_id["<EOS>"]

        self.question_id = tokenizer.token_to_id["<QUESTION>"]
        self.answer_id = tokenizer.token_to_id["<ANSWER>"]

    def build_sequence(self, question, answer):

        question_tokens = self.tokenizer.encode(question)
        answer_tokens = self.tokenizer.encode(answer)

        tokens = (
            [self.bos_id]
            + [self.question_id]
            + question_tokens
            + [self.answer_id]
            + answer_tokens
            + [self.eos_id]
        )

        return tokens

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        row = self.data.iloc[index]

        tokens = self.build_sequence(
            row["question"],
            row["reference_answer"]
        )

        # Truncate
        tokens = tokens[:self.max_length]

        # Input and target are shifted by one position.
        input_ids = tokens[:-1]
        target_ids = tokens[1:]

        # Padding
        padding_length = (
            self.max_length - 1 - len(input_ids)
        )

        input_ids += [self.pad_id] * padding_length
        target_ids += [self.pad_id] * padding_length

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),

            "target_ids": torch.tensor(
                target_ids,
                dtype=torch.long
            )
        }


def create_tokenizer(train_df):

    tokenizer = SimpleTokenizer()

    texts = (
        train_df["question"].tolist()
        + train_df["reference_answer"].tolist()
    )

    tokenizer.build_vocab(texts)

    return tokenizer

# Building the transformer

In [6]:
import math
import torch.nn as nn


class CausalSelfAttention(nn.Module):

    def __init__(self, d_model, num_heads, dropout):

        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv = nn.Linear(
            d_model,
            3 * d_model
        )

        self.output_projection = nn.Linear(
            d_model,
            d_model
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        batch_size, sequence_length, _ = x.shape

        qkv = self.qkv(x)

        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            sequence_length,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        attention_scores = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)

        # Causal mask
        mask = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                device=x.device
            )
        )

        attention_scores = attention_scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        attention_weights = torch.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        output = attention_weights @ v

        output = output.transpose(1, 2).contiguous()

        output = output.view(
            batch_size,
            sequence_length,
            self.d_model
        )

        return self.output_projection(output)


class FeedForward(nn.Module):

    def __init__(self, d_model, dropout):

        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)


class TransformerBlock(nn.Module):

    def __init__(
        self,
        d_model,
        num_heads,
        dropout
    ):

        super().__init__()

        self.layer_norm_1 = nn.LayerNorm(d_model)

        self.attention = CausalSelfAttention(
            d_model,
            num_heads,
            dropout
        )

        self.layer_norm_2 = nn.LayerNorm(d_model)

        self.feed_forward = FeedForward(
            d_model,
            dropout
        )

    def forward(self, x):

        x = x + self.attention(
            self.layer_norm_1(x)
        )

        x = x + self.feed_forward(
            self.layer_norm_2(x)
        )

        return x


class SmallLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        max_sequence_length,
        d_model=128,
        num_heads=4,
        num_layers=4,
        dropout=0.1
    ):

        super().__init__()

        self.vocab_size = vocab_size
        self.max_sequence_length = max_sequence_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            max_sequence_length,
            d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model,
                num_heads,
                dropout
            )
            for _ in range(num_layers)
        ])

        self.final_layer_norm = nn.LayerNorm(
            d_model
        )

        self.output_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Weight tying
        self.output_head.weight = (
            self.token_embedding.weight
        )

    def forward(self, input_ids):

        batch_size, sequence_length = input_ids.shape

        positions = torch.arange(
            sequence_length,
            device=input_ids.device
        )

        token_embeddings = self.token_embedding(
            input_ids
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = (
            token_embeddings
            + position_embeddings
        )

        for block in self.blocks:
            x = block(x)

        x = self.final_layer_norm(x)

        logits = self.output_head(x)

        return logits

# Creating the training script

In [7]:
import os
import random

import numpy as np
from torch.utils.data import DataLoader

In [21]:
# Configuration

checkpoint_dir = "checkpoints"

max_sequence_length_train = 128

batch_size_train = 4

d_model_train = 128

num_heads_train = 4

num_layers_train = 4

dropout_train = 0.1

learning_rate_train = 3e-4

weight_decay_train = 0.01

epochs_train = 100

seed_train = 42

In [9]:

# Reproducibility

random.seed(seed_train)

np.random.seed(seed_train)

torch.manual_seed(seed_train)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed_train)

In [15]:
# Device

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: cpu


In [11]:

# Load data

print("Training examples:", len(train_df))


Training examples: 43


In [34]:
# Build tokenizer

tokenizer = create_tokenizer(train_df)

print("Vocabulary size:", tokenizer.vocab_size)

os.makedirs(checkpoint_dir, exist_ok=True)

Vocabulary size: 420


In [14]:
# Create dataset

dataset = QADataset(
    dataframe=train_df,
    tokenizer=tokenizer,
    max_length=max_sequence_length_train
)


dataloader = DataLoader(
    dataset,
    batch_size=batch_size_train,
    shuffle=True
)

In [22]:
# Create model

model = SmallLanguageModel(
    vocab_size=tokenizer.vocab_size,
    max_sequence_length=max_sequence_length_train,
    d_model=d_model_train,
    num_heads=num_heads_train,
    num_layers=num_layers_train,
    dropout=dropout_train
).to(device)


print(
    "Model parameters:",
    sum(
        p.numel()
        for p in model.parameters()
    )
)

Model parameters: 863488


In [23]:
# Optimizer

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate_train,
    weight_decay=weight_decay_train
)

In [24]:
# Loss function

criterion = nn.CrossEntropyLoss(
    ignore_index=tokenizer.token_to_id["<PAD>"]
)

In [25]:
# Training

model.train()

for epoch in range(epochs_train):

    total_loss = 0.0

    for batch in dataloader:

        input_ids = batch["input_ids"].to(device)

        target_ids = batch["target_ids"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids)

        loss = criterion(
            logits.reshape(-1, tokenizer.vocab_size),
            target_ids.reshape(-1)
        )

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    average_loss = (
        total_loss / len(dataloader)
    )

    print(
        f"Epoch {epoch + 1:03d}/{epochs_train} "
        f"Loss: {average_loss:.4f}"
    )

Epoch 001/100 Loss: 71.9732
Epoch 002/100 Loss: 36.4907
Epoch 003/100 Loss: 22.6048
Epoch 004/100 Loss: 17.3160
Epoch 005/100 Loss: 14.4391
Epoch 006/100 Loss: 12.4754
Epoch 007/100 Loss: 11.1013
Epoch 008/100 Loss: 10.1605
Epoch 009/100 Loss: 9.4886
Epoch 010/100 Loss: 8.9482
Epoch 011/100 Loss: 8.4733
Epoch 012/100 Loss: 8.0015
Epoch 013/100 Loss: 7.7408
Epoch 014/100 Loss: 7.3673
Epoch 015/100 Loss: 7.1012
Epoch 016/100 Loss: 6.7971
Epoch 017/100 Loss: 6.5385
Epoch 018/100 Loss: 6.2730
Epoch 019/100 Loss: 5.9925
Epoch 020/100 Loss: 5.6554
Epoch 021/100 Loss: 5.4139
Epoch 022/100 Loss: 5.1867
Epoch 023/100 Loss: 4.9023
Epoch 024/100 Loss: 4.5792
Epoch 025/100 Loss: 4.2683
Epoch 026/100 Loss: 3.9249
Epoch 027/100 Loss: 3.5905
Epoch 028/100 Loss: 3.1684
Epoch 029/100 Loss: 2.8747
Epoch 030/100 Loss: 2.5478
Epoch 031/100 Loss: 2.2484
Epoch 032/100 Loss: 1.9150
Epoch 033/100 Loss: 1.6632
Epoch 034/100 Loss: 1.4900
Epoch 035/100 Loss: 1.2326
Epoch 036/100 Loss: 1.1255
Epoch 037/100 Loss: 

In [35]:
# Save checkpoint

checkpoint = {
    "model_state_dict": model.state_dict(),

    "vocab_size": tokenizer.vocab_size,

    "max_sequence_length": max_sequence_length_train,

    "d_model": d_model_train,

    "num_heads": num_heads_train,

    "num_layers": num_layers_train,

    "dropout": dropout_train
}

print()
print("Training complete.")
print("Saved tokenizer.")


Training complete.
Saved tokenizer.


# Adding validation

In [27]:
from sklearn.model_selection import train_test_split

In [29]:
df_1 = train_df.sample(
    frac=1,
    random_state=seed_train
).reset_index(drop=True)

split_index = int(0.9 * len(df_1))

train_df1 = df_1.iloc[:split_index]
validation_df = df_1.iloc[split_index:]

In [30]:
train_dataset = QADataset(
    train_df1,
    tokenizer,
    max_sequence_length_train
)

validation_dataset = QADataset(
    validation_df,
    tokenizer,
    max_sequence_length_train
)

In [31]:
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size_train,
    shuffle=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size_train
)

In [32]:
# Create validation loss function

def evaluate_loss(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(device)

            target_ids = batch["target_ids"].to(device)

            logits = model(input_ids)

            loss = criterion(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1)
            )

            total_loss += loss.item()

    model.train()

    return total_loss / len(dataloader)

In [33]:
validation_loss = evaluate_loss(
    model,
    validation_loader,
    criterion,
    device
)

print(
    f"Epoch {epoch + 1:03d} "
    f"Train Loss: {average_loss:.4f} "
    f"Validation Loss: {validation_loss:.4f}"
)

Epoch 100 Train Loss: 0.2483 Validation Loss: 0.1810


# Generating texts

In [36]:
# Recreate model

model = SmallLanguageModel(
    vocab_size=checkpoint["vocab_size"],
    max_sequence_length=checkpoint["max_sequence_length"],
    d_model=checkpoint["d_model"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(device)


model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

SmallLanguageModel(
  (token_embedding): Embedding(420, 128)
  (position_embedding): Embedding(128, 128)
  (blocks): ModuleList(
    (0-3): 4 x TransformerBlock(
      (layer_norm_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (attention): CausalSelfAttention(
        (qkv): Linear(in_features=128, out_features=384, bias=True)
        (output_projection): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (layer_norm_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (feed_forward): FeedForward(
        (network): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (final_layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (output_head): Linear(in_features=128, out_fea

In [52]:
# Generation

MAX_NEW_TOKENS = 50

def generate_answer(question):

    bos_id = tokenizer.token_to_id["<BOS>"]

    question_id = tokenizer.token_to_id["<QUESTION>"]

    answer_id = tokenizer.token_to_id["<ANSWER>"]

    eos_id = tokenizer.token_to_id["<EOS>"]

    question_tokens = tokenizer.encode(question)

    tokens = (
        [bos_id]
        + [question_id]
        + question_tokens
        + [answer_id]
    )

    for _ in range(MAX_NEW_TOKENS):

        input_tokens = tokens[
            -checkpoint["max_sequence_length"]:
        ]

        input_tensor = torch.tensor(
            [input_tokens],
            dtype=torch.long,
            device=device
        )

        with torch.no_grad():

            logits = model(input_tensor)

        next_token_logits = logits[
            0,
            -1
        ]

        # Greedy decoding
        next_token = torch.argmax(
            next_token_logits
        ).item()

        tokens.append(next_token)

        if next_token == eos_id:
            break

    # Only return tokens after <ANSWER>
    answer_start = tokens.index(answer_id) + 1

    answer_tokens = tokens[answer_start:]

    answer_tokens = [
        token
        for token in answer_tokens
        if token not in {
            tokenizer.token_to_id["<EOS>"],
            tokenizer.token_to_id["<PAD>"]
        }
    ]

    return tokenizer.decode(answer_tokens)

In [53]:
# Interactive interface

# while True:

    question = input("\nQuestion: ")

    if question.lower() in {
        "exit",
        "quit"
    }:
        break

    answer = generate_answer(question)

    # print("\nAnswer:", answer)


Question:  Having headache



Answer: at nine months — options ? <ANSWER> validate feelings duplicate products , reliever instructions , and growth checks .



Question:  Stop



Answer: at work — hygiene advice ? <ANSWER> cover coughs , wash hands , and stay home if febrile .


KeyboardInterrupt: Interrupted by user

# Add top-k sampling

In [47]:
def top_k_sampling(
    logits,
    k=10,
    temperature=1.0
):

    logits = logits / temperature

    values, indices = torch.topk(
        logits,
        k
    )

    probabilities = torch.softmax(
        values,
        dim=-1
    )

    selected = torch.multinomial(
        probabilities,
        num_samples=1
    )

    return indices[selected].item()

# Question evaluation

In [54]:
results = []


for _, row in test_df.iterrows():

    question = row["question"]

    answer = generate_answer(
        question
    )

    results.append({
        "QuestionId": row["QuestionId"],
        "question": question,
        "generated_answer": answer
    })


results_df = pd.DataFrame(results)


results_df.to_csv(
    "test_predictions.csv",
    index=False
)


print(results_df.to_string(index=False))

 QuestionId                                          question                                                                  generated_answer
       1001                 How is pregnancy anaemia managed?                                oral rehydration solution to prevent dehydration .
       1002            Fever in a two-month-old — what to do?                                         any persistent not exceed label maximum .
       1003         Pregnancy symptoms needing urgent review?  original containers , cool dry place , away from children , check expiry dates .
       1004      Teen refuses school citing panic — approach? usually postpone if moderate illness with routine screening , and growth checks .
       1005                 How can families prevent malaria?                                oral rehydration solution to prevent dehydration .
       1006  Child wheezing after dust exposure — plan items?                 validate feelings duplicate products , and stay home if fe

# Add Bleu style evaluation

In [55]:
from collections import Counter


def ngrams(tokens, n):

    return [
        tuple(tokens[i:i+n])
        for i in range(len(tokens) - n + 1)
    ]


def precision(reference, candidate, n=1):

    reference_ngrams = Counter(
        ngrams(reference, n)
    )

    candidate_ngrams = Counter(
        ngrams(candidate, n)
    )

    matches = 0

    total = sum(candidate_ngrams.values())

    for gram, count in candidate_ngrams.items():

        matches += min(
            count,
            reference_ngrams.get(gram, 0)
        )

    if total == 0:
        return 0

    return matches / total

In [58]:
# Add early stopping
best_validation_loss = float("inf")

patience = 10

epochs_without_improvement = 0

In [71]:
config = {
    "vocab_size": tokenizer.vocab_size,
    "max_sequence_length": max_sequence_length_train,
    "d_model": d_model_train,
    "num_heads": num_heads_train,
    "num_layers": num_layers_train,
    "dropout": dropout_train
}

In [72]:
import json

with open(
    "checkpoints/config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )